# AI Hub 문서요약 데이터 기반 LLM (Ollama) 요약 성능 평가

AI Hub의 '문서요약 텍스트' Validation 셋(신문기사)에 있는 **사람이 직접 작성한 생성요약문(Abstractive Summary)** 을 정답지(Ground Truth)로 삼아, 우리 서버에서 동작하는 **Ollama(LLM) 모델의 요약 생성 능력**을 측정합니다.\nAI Hub 정답지의 스타일(한 문장으로 핵심을 꾹꾹 눌러 담은 형태)을 모방하도록 특화된 프롬프트를 적용하여 평가를 진행합니다.

## 1. 라이브러리 및 데이터 로드

In [1]:
import pandas as pd
import numpy as np
import json
import zipfile
from rouge_score import rouge_scorer
import torch
from bert_score import BERTScorer
import ollama
import os
import warnings
warnings.filterwarnings('ignore')

# ZIP 파일 경로 설정
zip_path = '문서요약 텍스트/Validation/신문기사_valid_original.zip'
json_filename = 'valid_original.json'

# 샘플링할 문서의 갯수 (Ollama 추론 시간 고려, 50개면 보통 10~20분 소요)
SAMPLE_SIZE = 100

print("AI Hub JSON 데이터를 로딩합니다...")
documents = []
abstractive_summaries = []

with zipfile.ZipFile(zip_path, 'r') as z:
    with z.open(json_filename) as f:
        data = json.load(f)
        docs = data['documents']
        
        # 앞단에서 N개의 샘플만 추출
        import random
        sampled_docs = random.sample(docs, min(SAMPLE_SIZE, len(docs)))
        for doc in sampled_docs:
            # 본문 재조립 (paragraph 리스트들 파싱)
            full_text = ""
            for paragraph in doc['text']:
                for sentence_obj in paragraph:
                    full_text += sentence_obj['sentence'] + " "
            full_text = full_text.strip()
            
            # 사람이 작성한 생성 요약 추출
            abs_summary = doc['abstractive'][0]
            
            documents.append(full_text)
            abstractive_summaries.append(abs_summary)

df = pd.DataFrame({
    'context': documents,
    'aihub_summary': abstractive_summaries
})

print(f"총 {len(df)}개의 평가용 샘플 로드 완료.")
display(df.head(2))

AI Hub JSON 데이터를 로딩합니다...
총 100개의 평가용 샘플 로드 완료.


,context,aihub_summary
0,[헤럴드경제=황유진 기자] 미국 중앙은행인 연방준비제도(Fedㆍ연준)가 향후 경제성...,국 중앙은행인 연방준비제도는 저물가 지속ㆍ경기침체가 지속될 경우 금리인하를 고려하고...
1,한국뇌연구원은 한국뇌은행 네트워크(KBBN)를 통해 뇌질환 연구를 위한 사후 뇌기증...,22일 한국뇌연구원에 따르면 한국뇌은행이 100증례를 넘어선 네트워크를 통한 사후 ...


## 2. 특화된 프롬프트 탑재 요약기 세팅
AI Hub의 Abstractive Summary는 보통 **모든 핵심 내용이 '~(하)며, ~(이)라고 언급했다' 형태로 연결된 하나의 긴 문장**으로 구성되어 있습니다. 이를 LLM이 모방하도록 프롬프트를 튜닝했습니다.

In [ ]:
class OllamaEvaluator:
    def __init__(self, ollama_model="gemma3:27b"):
        self.ollama_model = os.getenv("OLLAMA_MODEL", ollama_model)
        
    def generate_summary(self, text):
        # AI Hub 정답지 스타일을 강제로 모방하도록 유도하는 특수 프롬프트
        prompt = f"""다음 기사를 읽고, 제공된 [작성 지침]에 따라 단 한 개의 완성된 문장으로만 전체 내용을 포괄하는 생성 요약문(Abstractive Summary)을 작성해라.

            [작성 지침]
            - 반드시 단 1개의 긴 문장(마침표 하나)으로 요약할 것.
            - 기호, 불릿 문자, 번호, 부사구 시작("이 기사는" 등)을 절대 사용하지 말 것.
            - 원문에서 여러 핵심 정보를 추출해 '~(하)며, ~(이)라고 점망했다/밝혔다' 형태의 자연스러운 연결 어미로 한 호흡에 이을 것.
            - 사람이 직접 주요 맥락을 새로 쓴 것처럼 매끄러운 평서문(-다) 하나로 마칠 것.

            [기사 본문]
            {text}"""
        try:
            response = ollama.chat(model=self.ollama_model, messages=[
                {'role': 'user', 'content': prompt},
            ])
            # 불필요한 따옴표나 공백 등 앞뒤 노이즈 컷팅
            result = response['message']['content'].strip()
            result = result.replace('\"', '').replace('\'', '')
            return result
        except Exception as e:
            return f"Error: {e}"

summarizer = OllamaEvaluator()

## 3. 대규모 요약 추론 실행
준비된 AI Hub 문서 텍스트를 LLM에 넣어 요약문을 생성합니다.

In [3]:
ollama_preds = []

print("LLM 요약문을 일괄 생성 중입니다... (1개당 수십 초 소요될 수 있음)")
total = len(df)
for idx, row in df.iterrows():
    text = row['context']
    
    # 요약 생성
    res = summarizer.generate_summary(text)
    ollama_preds.append(res)
    
    # 진행률 텍스트 표시
    if (idx + 1) % 5 == 0:
        print(f"진행 중... {idx+1}/{total}")

df['ollama_summary'] = ollama_preds
print("요약 생성 완료!")

# 결과 일부 눈으로 확인
display(df[['aihub_summary', 'ollama_summary']].head(3))

LLM 요약문을 일괄 생성 중입니다... (1개당 수십 초 소요될 수 있음)
진행 중... 5/100
진행 중... 10/100
진행 중... 15/100
진행 중... 20/100
진행 중... 25/100
진행 중... 30/100
진행 중... 35/100
진행 중... 40/100
진행 중... 45/100
진행 중... 50/100
진행 중... 55/100
진행 중... 60/100
진행 중... 65/100
진행 중... 70/100
진행 중... 75/100
진행 중... 80/100
진행 중... 85/100
진행 중... 90/100
진행 중... 95/100
진행 중... 100/100
요약 생성 완료!


,aihub_summary,ollama_summary
0,국 중앙은행인 연방준비제도는 저물가 지속ㆍ경기침체가 지속될 경우 금리인하를 고려하고...,연준 부의장은 경제성장 전망이 악화되거나 인플레이션이 목표치를 지속적으로 하회하고 ...
1,22일 한국뇌연구원에 따르면 한국뇌은행이 100증례를 넘어선 네트워크를 통한 사후 ...,한국뇌연구원은 한국뇌은행 네트워크를 통해 뇌질환 연구에 필수적인 사후 뇌기증이 10...
2,한국제약바이오협회가 10월 30일부터 18일 동안 미국과 유럽을 방문하여 선진화된 ...,"한국제약바이오협회는 원희목 회장 일행이 미국, 독일, 아일랜드, 영국 등 주요 바이..."


## 4. 평가 모델 세팅 (ROUGE & BERT)

In [4]:
# ROUGE Evaluator
rouge_evaluator = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# BERTScore Evaluator
print("BERTScore 모델 로딩 중...")
device = "cuda" if torch.cuda.is_available() else "cpu"
bert_scorer_model = BERTScorer(lang="ko", rescale_with_baseline=False, device=device)
print(f"BERTScore 준비 완료 (Device: {device})")

BERTScore 모델 로딩 중...
BERTScore 준비 완료 (Device: cuda)


## 5. 점수 산출 및 평균 비교

In [5]:
results = {
    'rouge1': [], 'rouge2': [], 'rougeL': [],
    'bert_p': [], 'bert_r': [], 'bert_f1': []
}

total = len(df)
print("스코어 측정 시작...")
for idx, row in df.iterrows():
    ref = str(row['aihub_summary'])
    cand = str(row['ollama_summary'])
    
    # ROUGE
    r_scores = rouge_evaluator.score(ref, cand)
    results['rouge1'].append(r_scores['rouge1'].fmeasure)
    results['rouge2'].append(r_scores['rouge2'].fmeasure)
    results['rougeL'].append(r_scores['rougeL'].fmeasure)
    
    # BERTScore
    p, r, f1 = bert_scorer_model.score([cand], [ref])
    results['bert_p'].append(p.item())
    results['bert_r'].append(r.item())
    results['bert_f1'].append(f1.item())
        
print("스코어 측정 완료!\n")

# 표 형태 출력
print("="*60)
print(f"[AI Hub 기준 LLM 단독 평가 (총 {total} 개 샘플)]")
print("============================================================")
print(f"{'Metric':<15} | {'Average Score':<15}")
print("-" * 60)

print_metrics = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'BERT-P', 'BERT-R', 'BERT-F1']
internal_metrics = ['rouge1', 'rouge2', 'rougeL', 'bert_p', 'bert_r', 'bert_f1']

for p_metric, i_metric in zip(print_metrics, internal_metrics):
    val = np.mean(results[i_metric])
    print(f"{p_metric:<15} | {val:.4f}")

print("============================================================")

스코어 측정 시작...
스코어 측정 완료!

[AI Hub 기준 LLM 단독 평가 (총 100 개 샘플)]
Metric          | Average Score  
------------------------------------------------------------
ROUGE-1         | 0.3259
ROUGE-2         | 0.0964
ROUGE-L         | 0.2928
BERT-P          | 0.7295
BERT-R          | 0.7786
BERT-F1         | 0.7530
